In [2]:
#PARTE 1.1 Revisar y describir el dataset:
import pandas as pd

# Cargar datos
df = pd.read_csv('https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/netflix_titles.csv')

n_filas, n_columnas = df.shape
print(f"El dataset tiene {n_filas} filas y {n_columnas} columnas.\n")

print("Tipos de datos por columna:")
print(df.dtypes)
print()

nulos = df.isnull().sum()
porcentaje_nulos = (nulos / n_filas * 100).round(2)

resumen_nulos = pd.DataFrame({
    'nulos': nulos,
    'porcentaje (%)': porcentaje_nulos
}).sort_values(by='nulos', ascending=False)

print("Valores nulos por columna (ordenados de mayor a menor):")
print(resumen_nulos)


El dataset tiene 8807 filas y 12 columnas.

Tipos de datos por columna:
show_id         object
type            object
title           object
director        object
cast            object
country         object
date_added      object
release_year     int64
rating          object
duration        object
listed_in       object
description     object
dtype: object

Valores nulos por columna (ordenados de mayor a menor):
              nulos  porcentaje (%)
director       2634           29.91
country         831            9.44
cast            825            9.37
date_added       10            0.11
rating            4            0.05
duration          3            0.03
show_id           0            0.00
type              0            0.00
title             0            0.00
release_year      0            0.00
listed_in         0            0.00
description       0            0.00


In [3]:
#PARTE 1.2 Transformar la columna date_added a tipo fecha.

df['date_added'] = pd.to_datetime(df['date_added'].str.strip(), errors='coerce')

print("Tipo de dato de date_added ahora:", df['date_added'].dtype)
print("\n¿Cuántas fechas no se pudieron convertir (quedaron como NaT)?")
print(df['date_added'].isnull().sum())

df[['title', 'date_added']].head()


Tipo de dato de date_added ahora: datetime64[ns]

¿Cuántas fechas no se pudieron convertir (quedaron como NaT)?
10


,title,date_added
0,Dick Johnson Is Dead,2021-09-25
1,Blood & Water,2021-09-24
2,Ganglands,2021-09-24
3,Jailbirds New Orleans,2021-09-24
4,Kota Factory,2021-09-24


In [4]:
#PARTE 1.3 Crear columnas auxiliares con assign:

df = df.assign(
    year_added=df['date_added'].dt.year,
    month_added=df['date_added'].dt.month
)
print("Nuevas columnas creadas: year_added, month_added\n")
df[['title', 'date_added', 'year_added', 'month_added']].head(10)

Nuevas columnas creadas: year_added, month_added



,title,date_added,year_added,month_added
0,Dick Johnson Is Dead,2021-09-25,2021.0,9.0
1,Blood & Water,2021-09-24,2021.0,9.0
2,Ganglands,2021-09-24,2021.0,9.0
3,Jailbirds New Orleans,2021-09-24,2021.0,9.0
4,Kota Factory,2021-09-24,2021.0,9.0
5,Midnight Mass,2021-09-24,2021.0,9.0
6,My Little Pony: A New Generation,2021-09-24,2021.0,9.0
7,Sankofa,2021-09-24,2021.0,9.0
8,The Great British Baking Show,2021-09-24,2021.0,9.0
9,The Starling,2021-09-24,2021.0,9.0


In [6]:
#PARTE 2.4 Utilizar .loc para seleccionar películas (type == 'Movie') que fueron agregadas después del año 2018.

peliculas_post_2018 = df.loc[
    (df['type'] == 'Movie') & (df['year_added'] > 2018)
]

print(f"Cantidad de películas agregadas después de 2018: {len(peliculas_post_2018)}")
peliculas_post_2018[['title', 'type', 'year_added', 'release_year']].head(10)

Cantidad de películas agregadas después de 2018: 3701


,title,type,year_added,release_year
0,Dick Johnson Is Dead,Movie,2021.0,2020
6,My Little Pony: A New Generation,Movie,2021.0,2021
7,Sankofa,Movie,2021.0,1993
9,The Starling,Movie,2021.0,2021
12,Je Suis Karl,Movie,2021.0,2021
13,Confessions of an Invisible Girl,Movie,2021.0,2021
16,Europe's Most Dangerous Man: Otto Skorzeny in ...,Movie,2021.0,2020
18,Intrusion,Movie,2021.0,2021
22,Avvai Shanmughi,Movie,2021.0,1996
23,Go! Go! Cory Carson: Chrissy Takes the Wheel,Movie,2021.0,2021


In [8]:
#PARTE 2.5 AHORA ESTA Utilizar str.contains() y str.extract():
titulos_love = df.loc[
    df['title'].str.contains('love', case=False, na=False)
]

print(f"Cantidad de títulos que contienen 'love': {len(titulos_love)}")
display(titulos_love[['title', 'type']].head(10))

df = df.assign(
    duration_minutes=df['duration'].str.extract(r'(\d+)\s*min').astype(float)
)

print("\nDuración extraída para películas:")
display(df.loc[df['type'] == 'Movie', ['title', 'duration', 'duration_minutes']].head(10))

Cantidad de títulos que contienen 'love': 196


,title,type
25,Love on the Spectrum,TV Show
158,Love Don't Cost a Thing,Movie
159,Love in a Puff,Movie
206,"LSD: Love, Sex Aur Dhokha",Movie
227,Really Love,Movie
246,Man in Love,Movie
375,Resort to Love,Movie
402,The Last Letter From Your Lover,Movie
485,Lethal Love,Movie
506,This Little Love Of Mine,Movie



Duración extraída para películas:


,title,duration,duration_minutes
0,Dick Johnson Is Dead,90 min,90.0
6,My Little Pony: A New Generation,91 min,91.0
7,Sankofa,125 min,125.0
9,The Starling,104 min,104.0
12,Je Suis Karl,127 min,127.0
13,Confessions of an Invisible Girl,91 min,91.0
16,Europe's Most Dangerous Man: Otto Skorzeny in ...,67 min,67.0
18,Intrusion,94 min,94.0
22,Avvai Shanmughi,161 min,161.0
23,Go! Go! Cory Carson: Chrissy Takes the Wheel,61 min,61.0


In [9]:
#PARTE 2.6 Aplicar explode() sobre la columna listed_in para obtener una fila por cada género.

df_generos = df.assign(
    genre=df['listed_in'].str.split(',')
).explode('genre')

df_generos['genre'] = df_generos['genre'].str.strip()

print(f"Filas originales: {len(df)}")
print(f"Filas después de explode(): {len(df_generos)}")

df_generos[['title', 'listed_in', 'genre']].head(10)


Filas originales: 8807
Filas después de explode(): 19323


,title,listed_in,genre
0,Dick Johnson Is Dead,Documentaries,Documentaries
1,Blood & Water,"International TV Shows, TV Dramas, TV Mysteries",International TV Shows
1,Blood & Water,"International TV Shows, TV Dramas, TV Mysteries",TV Dramas
1,Blood & Water,"International TV Shows, TV Dramas, TV Mysteries",TV Mysteries
2,Ganglands,"Crime TV Shows, International TV Shows, TV Act...",Crime TV Shows
2,Ganglands,"Crime TV Shows, International TV Shows, TV Act...",International TV Shows
2,Ganglands,"Crime TV Shows, International TV Shows, TV Act...",TV Action & Adventure
3,Jailbirds New Orleans,"Docuseries, Reality TV",Docuseries
3,Jailbirds New Orleans,"Docuseries, Reality TV",Reality TV
4,Kota Factory,"International TV Shows, Romantic TV Shows, TV ...",International TV Shows


In [10]:
#PARTE 2.7 Obtener un top 10 de géneros más frecuentes utilizando value_counts().
top10_generos = df_generos['genre'].value_counts().head(10)

print("Top 10 géneros más frecuentes en Netflix:\n")
print(top10_generos)

Top 10 géneros más frecuentes en Netflix:

genre
International Movies        2752
Dramas                      2427
Comedies                    1674
International TV Shows      1351
Documentaries                869
Action & Adventure           859
TV Dramas                    763
Independent Movies           756
Children & Family Movies     641
Romantic Movies              616
Name: count, dtype: int64


In [11]:
#PARTE 2.8 Aplicar where() y mask() para marcar las películas de más de 120 minutos como contenido largo en una nueva columna.

df['contenido_largo_where'] = pd.Series('Largo', index=df.index).where(
    df['duration_minutes'] > 120, other='Corto'
)

df['contenido_largo_mask'] = pd.Series('Corto', index=df.index).mask(
    df['duration_minutes'] > 120, other='Largo'
)

print("¿where() y mask() coinciden en todas las filas?",
      (df['contenido_largo_where'] == df['contenido_largo_mask']).all())

df.loc[df['type'] == 'Movie', ['title', 'duration_minutes', 'contenido_largo_where', 'contenido_largo_mask']].head(10)



¿where() y mask() coinciden en todas las filas? True


,title,duration_minutes,contenido_largo_where,contenido_largo_mask
0,Dick Johnson Is Dead,90.0,Corto,Corto
6,My Little Pony: A New Generation,91.0,Corto,Corto
7,Sankofa,125.0,Largo,Largo
9,The Starling,104.0,Corto,Corto
12,Je Suis Karl,127.0,Largo,Largo
13,Confessions of an Invisible Girl,91.0,Corto,Corto
16,Europe's Most Dangerous Man: Otto Skorzeny in ...,67.0,Corto,Corto
18,Intrusion,94.0,Corto,Corto
22,Avvai Shanmughi,161.0,Largo,Largo
23,Go! Go! Cory Carson: Chrissy Takes the Wheel,61.0,Corto,Corto


In [12]:
#PARTE 2.9 Utilizar .loc para filtrar películas que cumplen con:

peliculas_filtradas = df.loc[
    (df['duration_minutes'] > 100) &
    (df['rating'] == 'R') &
    (df['country'] == 'United States')
]

print(f"Cantidad de películas que cumplen las 3 condiciones: {len(peliculas_filtradas)}")
peliculas_filtradas[['title', 'duration_minutes', 'rating', 'country']].head(10)

Cantidad de películas que cumplen las 3 condiciones: 243


,title,duration_minutes,rating,country
48,Training Day,122.0,R,United States
81,Kate,106.0,R,United States
131,Blade Runner: The Final Cut,117.0,R,United States
139,Do the Right Thing,120.0,R,United States
144,House Party,104.0,R,United States
174,Tears of the Sun,121.0,R,United States
175,The Blue Lagoon,105.0,R,United States
178,The Interview,112.0,R,United States
183,In the Line of Fire,129.0,R,United States
247,Sweet Girl,110.0,R,United States


In [13]:
#PARTE 2.10 Utilizar .style para formatear visualmente el top 10 de películas más largas.
top10_largas = (
    df.loc[df['type'] == 'Movie', ['title', 'duration_minutes', 'rating', 'country']]
    .sort_values(by='duration_minutes', ascending=False)
    .head(10)
    .reset_index(drop=True)
)

top10_largas.style \
    .background_gradient(subset=['duration_minutes'], cmap='Reds') \
    .format({'duration_minutes': '{:.0f} min'}) \
    .set_caption('🎬 Top 10 películas más largas de Netflix') \
    .set_properties(**{'text-align': 'left'})


,title,duration_minutes,rating,country
0,Black Mirror: Bandersnatch,312 min,TV-MA,United States
1,Headspace: Unwind Your Mind,273 min,TV-G,nan
2,The School of Mischief,253 min,TV-14,Egypt
3,No Longer kids,237 min,TV-14,Egypt
4,Lock Your Girls In,233 min,TV-PG,nan
5,Raya and Sakina,230 min,TV-14,nan
6,Once Upon a Time in America,229 min,R,"Italy, United States"
7,Sangam,228 min,TV-14,India
8,Lagaan,224 min,PG,"India, United Kingdom"
9,Jodhaa Akbar,214 min,TV-14,India


In [14]:
#PARTE 2.11 ¿Cuáles son las combinaciones más frecuentes de género y rating en el dataset? (Sugerencia: utilizar value_counts con subset=["genre", "rating"] después de aplicar explode()).

top10_genero_rating = (
    df_generos[['genre', 'rating']]
    .value_counts()
    .head(10)
)

print("Top 10 combinaciones más frecuentes de género y rating:\n")
print(top10_genero_rating)


Top 10 combinaciones más frecuentes de género y rating:

genre                   rating
International Movies    TV-MA     1130
                        TV-14     1065
Dramas                  TV-MA      830
International TV Shows  TV-MA      714
Dramas                  TV-14      693
International TV Shows  TV-14      472
Comedies                TV-14      465
TV Dramas               TV-MA      434
Comedies                TV-MA      431
Dramas                  R          375
Name: count, dtype: int64


In [16]:
#PARTE 12

titulos_anios_unicos = df[['title', 'release_year']].drop_duplicates()


titulos_duplicados_mask = titulos_anios_unicos['title'].duplicated(keep=False)

titulos_con_multiples_anios = (
    titulos_anios_unicos.loc[titulos_duplicados_mask]
    .sort_values(by='title')
)

print(f"Cantidad de títulos con más de un release_year distinto: "
      f"{titulos_con_multiples_anios['title'].nunique()}")

titulos_con_multiples_anios.head(15)

Cantidad de títulos con más de un release_year distinto: 0


,title,release_year


In [17]:
#PARTE 13 ¿Cuántos títulos únicos hay en total en la columna title?

total_filas = len(df)
titulos_unicos = df['title'].nunique()
titulos_repetidos = total_filas - titulos_unicos

print(f"Total de filas en el dataset: {total_filas}")
print(f"Cantidad de títulos únicos: {titulos_unicos}")
print(f"Diferencia (títulos que se repiten al menos una vez): {titulos_repetidos}")


Total de filas en el dataset: 8807
Cantidad de títulos únicos: 8807
Diferencia (títulos que se repiten al menos una vez): 0
